In [ ]:
import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from IPython.display import display

from tqdm import tqdm

import SimpleITK as sitk

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy import stats

np.set_printoptions(precision=4)

from brainpred import paths, splits
from brainpred.utils import display as vis
from brainpred.utils import intensity, metrics, ml as uml
from brainpred.features.longitudinal import whole_brain_based as features_loader_longitudinal_whole_brain_based
from brainpred.features.longitudinal import tissue_based as features_loader_longitudinal_tissue_based
from brainpred.features.cross_sectional import tissue_based as features_loader_cross_sectional_tissue_based
from brainpred.features import demographic as features_loader_demographic

# Load features

## Demographic features

In [ ]:
loader_demographic  = features_loader_demographic.FeaturesLoader
features_path_name_demographic = paths.get_work_folder("features", "demographic", "demographic")

reader_demographic = loader_demographic()
data_demographic = reader_demographic.get_df(
                     return_patient_id=True, 
                     remove_patients=splits.get_excluded_patients())

data_ids_demographic = data_demographic.pop('patient_id').tolist()
data_columns_demographic = data_demographic.columns.tolist()
display(data_demographic.head())

vis.plot_data_voxplot(data_demographic, data_columns_demographic, figsize=(8,4))

## Longitudinal features

### Whole brain based

In [ ]:
loader_logitudinal_wbb = features_loader_longitudinal_whole_brain_based.FeaturesLoader
features_path_name_logitudinal_wbb = paths.get_work_folder("features", "longitudinal", "whole_brain_based", "whole_brains_stats")
columns = None

reader_logitudinal_wbb = loader_logitudinal_wbb()
data_logitudinal_wbb = reader_logitudinal_wbb.get_df(columns=columns,
                     remove_columns=False, 
                     return_patient_id=True, 
                     id_patients=data_ids_demographic,
                     remove_patients=False)

data_ids_logitudinal_wbb = data_logitudinal_wbb.pop('patient_id').astype(int).tolist()
data_columns_logitudinal_wbb = data_logitudinal_wbb.columns.tolist()

print(data_logitudinal_wbb.shape)
display(data_logitudinal_wbb.head())
vis.plot_data_voxplot(data_logitudinal_wbb, data_columns_logitudinal_wbb, figsize=(8,4))



### Tissue based

In [ ]:
IMPORTANT_TISSUES = [43, 4, 53, 17, 49, 10, 42, 3, 2006, 1006, 2009, 1009, 2029, 1029, 2027, 1027, 2028, 1028, 2025, 1025, 2005, 1005, 14]
# IMPORTANT_TISSUES = None
loader_logitudinal_tb = features_loader_longitudinal_tissue_based.FeaturesLoader
nb_tissues = 96
metrics_name = ['vol_fraction']
matching_ids = splits.get_matching_ids()


reader_logitudinal_tb = loader_logitudinal_tb(nb_tissues, matching_ids)
data_logitudinal_tb = reader_logitudinal_tb.get_df(tissue_ids=IMPORTANT_TISSUES,
                                                   metrics_name=metrics_name,
                                                   remove_tissues=False,
                                                   remove_metrics=False,
                                                    return_patient_id=True, 
                                                    id_patients=data_ids_demographic,
                                                    remove_patients=False)

data_ids_logitudinal_tb = data_logitudinal_tb.pop('patient_id').astype(int).tolist()
data_columns_logitudinal_tb = data_logitudinal_tb.columns.tolist()

print(data_logitudinal_tb.shape)
display(data_logitudinal_tb.head())
vis.plot_data_voxplot(data_logitudinal_tb, data_columns_logitudinal_tb, figsize=(50,10))



## Cross-sectional features

### Tissue based

In [ ]:
IMPORTANT_TISSUES = [43, 4, 53, 17, 49, 10, 42, 3, 2006, 1006, 2009, 1009, 2029, 1029, 2027, 1027, 2028, 1028, 2025, 1025, 2005, 1005, 14]
IMPORTANT_TISSUES = None
loader_cross_sectional_tb = features_loader_cross_sectional_tissue_based.FeaturesLoader
nb_tissues = 96
metrics_name = ['vol_hunt3']
matching_ids = splits.get_matching_ids()


reader_cross_sectional_tb = loader_cross_sectional_tb(nb_tissues, matching_ids)
data_cross_sectional_tb = reader_cross_sectional_tb.get_df(tissue_ids=IMPORTANT_TISSUES,
                                                   metrics_name=metrics_name,
                                                   remove_tissues=False,
                                                   remove_metrics=False,
                                                    return_patient_id=True, 
                                                    id_patients=data_ids_demographic,
                                                    remove_patients=False)

data_ids_cross_sectional_tb = data_cross_sectional_tb.pop('patient_id').astype(int).tolist()
data_columns_cross_sectional_tb = data_cross_sectional_tb.columns.tolist()

print(data_cross_sectional_tb.shape)
display(data_cross_sectional_tb.head())
vis.plot_data_voxplot(data_cross_sectional_tb, data_columns_cross_sectional_tb, figsize=(50,10))



# Correlation

In [ ]:
# data_logitudinal_wbb["age_hunt3"] = data_demographic["age_hunt3"]
# data_logitudinal_wbb["age_hunt4"] = data_demographic["age_hunt4"]
# data = pd.concat([data_demographic, data_logitudinal_wbb], axis=1)

data = pd.concat([data_demographic, data_logitudinal_wbb, data_cross_sectional_tb], axis=1)
corr_df, to_drop = uml.correlation_df(data, corr_th=0.9)

print(f"High correlated: {to_drop}")
# vis.plot_data_precomputed_correlation_df(corr_df, figsize=(70, 60))
vis.plot_data_precomputed_correlation_df(corr_df, figsize=(150, 100))
# vis.plot_data_precomputed_correlation_df(corr_df, figsize=(20, 20))


In [ ]:
vis.plot_2D_points(data_demographic["age_hunt3"], data_demographic["age_hunt4"], "age_hunt3", "age_hunt4", figsize=(8,4))

vis.plot_2D_points(data_demographic["age_hunt3"], data_logitudinal_wbb["bpf_hunt3"], "age_hunt3", "bpf_hunt3", figsize=(8,4))
vis.plot_2D_points(data_demographic["age_hunt4"], data_logitudinal_wbb["bpf_hunt4"], "age_hunt4", "bpf_hunt4", figsize=(8,4))

vis.plot_2D_points(data_demographic["age_hunt3"], data_logitudinal_wbb["bpf_hunt4"], "age_hunt3", "bpf_hunt4", figsize=(8,4))
vis.plot_2D_points(data_demographic["age_hunt4"], data_logitudinal_wbb["bpf_hunt3"], "age_hunt4", "bpf_hunt3", figsize=(8,4))


# Preprocessing

In [ ]:
# def remove_outliers_zscore(data, threshold=3):
#     z_scores = stats.zscore(data)
#     abs_z_scores = np.abs(z_scores)
#     filtered_data = data[(abs_z_scores < threshold).all(axis=1)]
#     return filtered_data

def remove_outliers_zscore(data, threshold=3):
    z_scores = stats.zscore(data)
    abs_z_scores = np.abs(z_scores)
    filtered_indices = np.where((abs_z_scores >= threshold).any(axis=1))[0]
    filtered_data = data[(abs_z_scores < threshold).all(axis=1)]
    removed_indices = np.delete(np.arange(len(data)), filtered_indices)
    return filtered_data, removed_indices

def standarize_and_normalize(data):
    scaler = StandardScaler()
    data_standardized = scaler.fit_transform(data)

    normalizer = MinMaxScaler()
    data_normalized = normalizer.fit_transform(data_standardized)
    return data_normalized

In [ ]:
data_logitudinal_wbb_prep, removed_indices = remove_outliers_zscore(data_logitudinal_wbb, 2.5)
data_logitudinal_wbb_prep = standarize_and_normalize(data_logitudinal_wbb_prep)
data_logitudinal_wbb_prep_df = pd.DataFrame(data_logitudinal_wbb_prep, columns=data_columns_logitudinal_wbb)


vis.plot_data_voxplot(data_logitudinal_wbb_prep, data_columns_logitudinal_wbb, figsize=(8,4))
print(data_logitudinal_wbb_prep_df.shape)
print(removed_indices.shape)